In [1]:
from typing import Union
import pandas as pd
import numpy as np

import pymc as pm
import pymc.sampling.jax as pmjax
import pytensor.tensor as pt
import arviz as az
import arviz_stats as azs
import arviz_plots as azp
import xarray as xr

import matplotlib.pyplot as plt

RANDOM_SEED = 694973
np.random.seed(RANDOM_SEED)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# for reproducibility
print("pandas: "+pd.__version__)
print("numpy: "+np.__version__)
print("pymc: "+pm.__version__)
print("arviz: "+az.__version__)

pandas: 3.0.1
numpy: 2.4.3
pymc: 6.2.0
arviz: 1.2.0


In [2]:
df = pd.read_csv('data/projection_data.csv')
df.head(10)

,player_full_id,SEASON,shot_type,PLAYER_ID,PLAYER,player_age,age_z,HEIGHT_INCHES,height_z,birthdate,posit,attempts,actual_makes,expected_makes,offset
0,A.J. Lawson (1630639),2022-23,DUNK,1630639.0,A.J. Lawson,23.041752,-1.047506,78.0,-0.203215,2000-07-15,G,6,5,4.606923,1.196045
1,A.J. Lawson (1630639),2022-23,JUMPER,1630639.0,A.J. Lawson,23.041752,-1.047506,78.0,-0.203215,2000-07-15,G,27,12,10.200240,-0.498953
2,A.J. Lawson (1630639),2022-23,LAYUP,1630639.0,A.J. Lawson,23.041752,-1.047506,78.0,-0.203215,2000-07-15,G,11,5,6.662227,0.429093
3,A.J. Lawson (1630639),2023-24,DUNK,1630639.0,A.J. Lawson,24.043806,-0.804040,78.0,-0.203215,2000-07-15,G,13,12,9.504221,1.000180
4,A.J. Lawson (1630639),2023-24,JUMPER,1630639.0,A.J. Lawson,24.043806,-0.804040,78.0,-0.203215,2000-07-15,G,62,15,23.988975,-0.460282
5,A.J. Lawson (1630639),2023-24,LAYUP,1630639.0,A.J. Lawson,24.043806,-0.804040,78.0,-0.203215,2000-07-15,G,46,27,28.212996,0.461315
6,A.J. Lawson (1630639),2024-25,DUNK,1630639.0,A.J. Lawson,25.043121,-0.561239,78.0,-0.203215,2000-07-15,G,14,11,9.627423,0.789263
7,A.J. Lawson (1630639),2024-25,JUMPER,1630639.0,A.J. Lawson,25.043121,-0.561239,78.0,-0.203215,2000-07-15,G,106,33,39.452497,-0.522819
8,A.J. Lawson (1630639),2024-25,LAYUP,1630639.0,A.J. Lawson,25.043121,-0.561239,78.0,-0.203215,2000-07-15,G,70,36,41.499993,0.375789
9,A.J. Lawson (1630639),2025-26,DUNK,1630639.0,A.J. Lawson,26.042437,-0.318437,78.0,-0.203215,2000-07-15,G,4,3,2.848839,0.906141


In [ ]:
# df = df.loc[df['SEASON'].isin(['2021-22','2022-23','2023-24','2024-25','2025-26'])]
# min_attempts_threshold = 100
# player_attempts = df.groupby("player_full_id")["attempts"].sum()
# active_players = player_attempts[player_attempts >= min_attempts_threshold].index
# df_filtered = df[df["player_full_id"].isin(active_players)]
# df_filtered = df_filtered.groupby(['player_full_id', 'shot_type']).filter(lambda x: x['attempts'].sum() >= 20)
# df_filtered.head()
# df = df_filtered.copy()

In [4]:
df.shape

(7532, 15)

In [5]:
len(df['player_full_id'].unique())

720

In [6]:
def define_index(data: pd.DataFrame, label: str) -> Union[np.array, dict]:
    """Defines an index variable starting at 0. 
    Args:
        data (pd.DataFrame): dataframe with values to index
        label (str): column name user wishes to index in string format

    Returns:
        Union[
            np.array: indexed values
            dict: dictionary mapping the input values and their indexed values
            ]
    """
    unq_ids = data[label].astype(str).unique()
    lookup = {v: i for i, v in enumerate(unq_ids)}
    index_vals = data[label].astype(str).map(lookup).values
    return index_vals, lookup

In [7]:
# These are some generic helper functions to help sample for the model
def sample(model: pm.Model, draws: int = 2000, tune: int = 2000, chains: int = 4, target_accept: float = 0.99, random_seed: int = RANDOM_SEED, path:str = 'temp.nc', **kwargs):
    """
    Fit model using MCMC.

    Parameters
    ----------
    model: pm.Model
        PyMC model object.
    draws : int
        Number of draws to keep from the sampling process.
    tune : int
        Number of tuning steps to take before sampling.
    chains : int
        Number of chains to sample.
    target_accept : float
        Target acceptance probability for step size adaptation.
    random_seed : int
        Seed for randomness.
    """
    with model:
        trace = pmjax.sample_numpyro_nuts(
            draws=draws,
            tune=tune,
            chains=chains,
            target_accept=target_accept,
            random_seed=random_seed,
            idata_kwargs={"log_likelihood": False}
        )
    try:
        xr.DataTree.to_netcdf(trace, path)
    except Exception as e:
        print(f"Error saving trace to {path}: {e}")
    return trace

def compute_log_likelihood(model: pm.Model, trace: az.InferenceData) -> None:
    """Wrapper to compute elemwise log_likelihood of model given InferenceData with posterior group
    Args:
        model (pm.Model): A PyMC model object
        trace (az.InferenceData): Results from sampling
    """
    with model:
        pm.compute_log_likelihood(trace)
    return None

def sample_posterior_pred(model: pm.Model, trace: az.InferenceData) -> az.InferenceData:
    """Generates samples from the posterior predictive distribution for model checks

    Args:
        model (pm.Model): A PyMC model object
        trace (az.InferenceData): Results from sampling

    Returns:
        az.InferenceData: An ArviZ InferenceData object containing the posterior predictive samples.
    """
    with model:
        spp = pm.sample_posterior_predictive(
            trace,
            extend_inferencedata=True,
            random_seed=RANDOM_SEED,
        )
    return spp

In [8]:
df["player_idx"], player_lookup = define_index(df, "player_full_id")
player_idx_vals = df['player_idx'].values.astype("int32")
n_obs = len(df)

df['season_idx'], season_lookup = define_index(df, "SEASON")
season_idx_vals = df['season_idx'].values.astype("int32")

df['shot_type_idx'], shot_type_lookup = define_index(df, "shot_type")
shot_type_idx_vals = df['shot_type_idx'].values.astype("int32")

df['position_idx'], position_lookup = define_index(df, "posit")
position_idx_vals = df['position_idx'].values.astype("int32")

target = df['actual_makes'].values
offset_vals = df['offset'].values
age_z_vals = df['age_z'].values
height_z_vals = df['height_z'].values
attempts_vals = df['attempts'].values
expected_makes_vals = df['expected_makes'].values

In [9]:
season_list = sorted(df['SEASON'].unique())
season_list

['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']

In [ ]:
# final form of v1
coords = {
    "obs_id": np.arange(len(target)),
    "shot_type":shot_type_lookup.keys(),
    "player": player_lookup.keys(),
    "season" : season_list,
    "season_walk" : season_list[1:],
    }

with pm.Model(coords=coords) as model:
    shots_made = pm.Data("shots_made", target, dims=("obs_id",))
    offset_data = pm.Data("offset_data", offset_vals, dims=("obs_id",))
    attempts_data = pm.Data("attempts_data", attempts_vals, dims=("obs_id",))
    player_idx = pm.Data("player_idx", player_idx_vals, dims=("obs_id",))
    season_idx = pm.Data("season_idx", season_idx_vals, dims=("obs_id",))
    shot_type_idx = pm.Data("shot_type_idx", shot_type_idx_vals, dims=("obs_id",))

    # baseline
    offset_weight = pm.Normal("offset_weight", mu=1.0, sigma=0.25, dims=("shot_type",))
    shot_type_intercept = pm.Normal("shot_type_intercept", mu=0, sigma=1, dims=("shot_type",))
    # career talent
    sigma_career = pm.HalfNormal("sigma_career", sigma=1.0, dims=("shot_type",))
    player_mean_dev = pm.Normal("player_mean_dev", mu=0.0, sigma=1.0, dims=("player", "shot_type"))
    player_mean_centered = player_mean_dev - player_mean_dev.mean(axis=0)
    player_mean = pm.Deterministic(
        "player_mean",
        player_mean_centered * sigma_career[None, :], dims=("player", "shot_type")
        )
    # walk
    sigma_walk = pm.HalfNormal("sigma_walk", sigma=1.0, dims=("shot_type",))
    player_innovations = pm.Normal(
        "player_innovations",
        mu=0.0, sigma=1.0,
        dims=("season_walk", "player", "shot_type")
    )
    raw_walk = pt.concatenate([
       pt.zeros((1, len(coords["player"]), len(coords["shot_type"]))),
       pt.cumsum(player_innovations, axis=0)
    ], axis=0)
    player_walk_centered = raw_walk - raw_walk.mean(axis=0, keepdims=True)
    player_walk = player_walk_centered * sigma_walk[None, None, :]
    # player effect
    player_effect = pm.Deterministic(
        "player_effect",
        player_mean[None, :, :] + player_walk, dims=("season", "player", "shot_type")
        )
    player_contribution = player_effect[season_idx, player_idx, shot_type_idx]

    # expected value
    theta = shot_type_intercept[shot_type_idx] + offset_weight[shot_type_idx]*offset_data + player_contribution
    pm.Binomial("obs", n=attempts_data, p=pm.math.sigmoid(theta), observed=shots_made)

In [ ]:
trace = sample(model, tune=1000, draws=1000, target_accept=0.9, path='skill_trace.nc')

In [ ]:
azs.summary(trace, var_names=["offset_weight","shot_type_intercept", "sigma_career", "sigma_walk"], ci_prob=0.95, round_to=2)

In [ ]:
player = azs.summary(trace, var_names=["player_mean", "player_effect"], ci_prob=0.95, round_to=2)
player.head(10)

In [ ]:
coords = {
    "obs_id": np.arange(len(target)),
    "shot_type":shot_type_lookup.keys(),
    "player": player_lookup.keys(),
    "season": season_list,
    }
n_shot_types = len(shot_type_lookup.keys())
n_seasons = len(coords["season"])

with pm.Model(coords=coords) as model:
    shots_made = pm.Data("shots_made", target, dims=("obs_id",))
    offset_data = pm.Data("offset_data", offset_vals, dims=("obs_id",))
    attempts_data = pm.Data("attempts_data", attempts_vals, dims=("obs_id",))
    player_idx = pm.Data("player_idx", player_idx_vals, dims=("obs_id",))
    season_idx = pm.Data("season_idx", season_idx_vals, dims=("obs_id",))
    shot_type_idx = pm.Data("shot_type_idx", shot_type_idx_vals, dims=("obs_id",))

    # baseline
    offset_weight = pm.Normal("offset_weight", mu=0.2, sigma=0.2, dims=("shot_type",))
    shot_type_intercept = pm.Normal("shot_type_intercept", mu=0, sigma=1, dims=("shot_type",))
    # career skill
    player_st_dev = pm.HalfNormal("player_st_dev", 0.5, dims=("shot_type",))
    z_mu = pm.Normal("z_mu", mu=0, sigma=1, dims=("shot_type", "player"))
    mu = pm.Deterministic("mu", z_mu.T * player_st_dev[None, :], dims=("player", "shot_type"))
    # seasonal component
    sigma_dev = pm.Exponential("sigma_dev", 1.0, dims="shot_type") # season deviation
    delta_raw = pm.ZeroSumNormal(
        "delta_raw",
        sigma=1.0,
        dims=("player", "shot_type", "season"),
        n_zerosum_axes=1
    )
    delta = pm.Deterministic(
        "delta",
        delta_raw * sigma_dev[None, :, None],
        dims=("player", "shot_type", "season")
    )
    skill_grid = mu[:, :, None] + delta
    skill_obs = skill_grid[player_idx, shot_type_idx, season_idx]
    
    theta = (
        offset_weight[shot_type_idx] * offset_data
        + shot_type_intercept[shot_type_idx]
        + skill_obs
    )
    pm.Binomial("obs", n=attempts_data, p=pm.math.sigmoid(theta), observed=shots_made)

In [47]:
trace = sample(model, tune=1000, draws=1000, target_accept=0.9, dense_mass=True)

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


In [48]:
azs.summary(trace, var_names=["offset_weight", "shot_type_intercept", "player_st_dev", "sigma_dev"], ci_prob=0.95, round_to=2)

,mean,sd,eti95_lb,eti95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
offset_weight[DUNK],0.35,0.03,0.29,0.40,3102.33,3071.59,1.00,0.0,0.0
offset_weight[JUMPER],0.21,0.04,0.13,0.29,1407.41,2108.87,1.00,0.0,0.0
offset_weight[LAYUP],0.11,0.02,0.08,0.15,2781.55,2780.91,1.00,0.0,0.0
offset_weight[HOOK],0.17,0.07,0.03,0.31,4751.36,2909.74,1.00,0.0,0.0
shot_type_intercept[DUNK],1.72,0.04,1.65,1.79,2433.04,2794.22,1.00,0.0,0.0
shot_type_intercept[JUMPER],-0.49,0.02,-0.52,-0.45,1308.38,1971.09,1.00,0.0,0.0
shot_type_intercept[LAYUP],0.13,0.01,0.11,0.16,2312.91,2425.23,1.00,0.0,0.0
shot_type_intercept[HOOK],-0.11,0.01,-0.14,-0.08,2500.52,2905.48,1.00,0.0,0.0
player_st_dev[DUNK],0.38,0.02,0.35,0.41,1244.47,2046.69,1.00,0.0,0.0
player_st_dev[JUMPER],0.16,0.00,0.15,0.17,1047.90,1635.64,1.00,0.0,0.0


In [26]:
coords = {
    "obs_id": np.arange(len(target)),
    "shot_type":shot_type_lookup.keys(),
    "player": player_lookup.keys(),
    "season": season_list,
    }
n_shot_types = len(shot_type_lookup.keys())
n_seasons = len(coords["season"])

with pm.Model(coords=coords) as model_chol:
    shots_made = pm.Data("shots_made", target, dims=("obs_id",))
    offset_data = pm.Data("offset_data", offset_vals, dims=("obs_id",))
    attempts_data = pm.Data("attempts_data", attempts_vals, dims=("obs_id",))
    player_idx = pm.Data("player_idx", player_idx_vals, dims=("obs_id",))
    season_idx = pm.Data("season_idx", season_idx_vals, dims=("obs_id",))
    shot_type_idx = pm.Data("shot_type_idx", shot_type_idx_vals, dims=("obs_id",))

    # baseline
    offset_weight = pm.Normal("offset_weight", mu=0.2, sigma=0.2, dims=("shot_type",))
    shot_type_intercept = pm.Normal("shot_type_intercept", mu=0, sigma=1, dims=("shot_type",))
    # career skill
    chol, corr, stds = pm.LKJCholeskyCov("chol_shot_type", n=n_shot_types, eta=2.0, sd_dist=pm.HalfNormal.dist(0.25))
    mu = pm.MvNormal("mu", mu=pt.zeros(n_shot_types), chol=chol, dims=("player", "shot_type"))
    # seasonal component
    sigma_dev = pm.HalfNormal("sigma_dev", 0.25, dims="shot_type") # season deviation
    delta_raw = pm.ZeroSumNormal(
        "delta_raw",
        sigma=1.0,
        dims=("player", "shot_type", "season"),
        n_zerosum_axes=1
    )
    delta = pm.Deterministic(
        "delta",
        delta_raw * sigma_dev[None, :, None],
        dims=("player", "shot_type", "season")
    )
    skill_obs = mu[player_idx, shot_type_idx] + delta[player_idx, shot_type_idx, season_idx]
    
    theta = (
        offset_weight[shot_type_idx] * offset_data
        + shot_type_intercept[shot_type_idx]
        + skill_obs
    )
    pm.Binomial("obs", n=attempts_data, p=pm.math.sigmoid(theta), observed=shots_made)

In [27]:
trace_chol = sample(model_chol, tune=3000, draws=2000, target_accept=0.99, dense_mass=True)

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


In [32]:
azs.summary(trace_chol, var_names=["offset_weight", "shot_type_intercept", "chol_shot_type","sigma_dev"], ci_prob=0.95, round_to=2)

,mean,sd,eti95_lb,eti95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
offset_weight[DUNK],0.58,0.08,0.42,0.74,12563.87,6454.87,1.00,0.0,0.0
offset_weight[JUMPER],0.77,0.07,0.63,0.89,5664.12,5625.40,1.00,0.0,0.0
offset_weight[LAYUP],0.40,0.04,0.33,0.49,7761.59,5847.96,1.00,0.0,0.0
offset_weight[HOOK],0.26,0.14,-0.01,0.52,15783.96,6485.72,1.00,0.0,0.0
shot_type_intercept[DUNK],1.50,0.08,1.35,1.65,11243.88,6974.47,1.00,0.0,0.0
shot_type_intercept[JUMPER],-0.16,0.03,-0.22,-0.10,5195.33,5420.08,1.00,0.0,0.0
shot_type_intercept[LAYUP],0.04,0.02,0.00,0.08,5369.31,5992.48,1.00,0.0,0.0
shot_type_intercept[HOOK],-0.11,0.03,-0.16,-0.06,3257.40,4784.55,1.00,0.0,0.0
chol_shot_type[0],0.38,0.02,0.34,0.43,1873.32,3141.37,1.00,0.0,0.0
chol_shot_type[1],0.01,0.01,-0.01,0.03,2919.72,3955.24,1.00,0.0,0.0


In [36]:
az.summary(trace_chol, var_names=["chol_shot_type_corr","chol_shot_type_stds"], ci_prob=0.95, round_to=2)

/opt/homebrew/Caskroom/miniforge/base/envs/pie/lib/python3.12/site-packages/arviz_stats/base/diagnostics.py:90: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
/opt/homebrew/Caskroom/miniforge/base/envs/pie/lib/python3.12/site-packages/arviz_stats/base/diagnostics.py:313: RuntimeWarning: invalid value encountered in scalar divide
  varsd = varvar / evar / 4


,mean,sd,eti95_lb,eti95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
"chol_shot_type_corr[0, 0]",1.00,0.00,1.00,1.00,8000.00,8000.00,NaN,0.00,NaN
"chol_shot_type_corr[0, 1]",0.10,0.07,-0.04,0.25,2894.91,3979.98,1.00,0.00,0.0
"chol_shot_type_corr[0, 2]",0.49,0.06,0.37,0.60,2530.24,3854.26,1.00,0.00,0.0
"chol_shot_type_corr[0, 3]",0.21,0.11,-0.02,0.42,794.08,1435.65,1.00,0.00,0.0
"chol_shot_type_corr[1, 0]",0.10,0.07,-0.04,0.25,2894.91,3979.98,1.00,0.00,0.0
"chol_shot_type_corr[1, 1]",1.00,0.00,1.00,1.00,7875.47,7664.86,1.00,0.00,0.0
"chol_shot_type_corr[1, 2]",0.53,0.04,0.45,0.62,4434.43,5476.26,1.00,0.00,0.0
"chol_shot_type_corr[1, 3]",0.49,0.10,0.28,0.68,355.65,531.01,1.01,0.01,0.0
"chol_shot_type_corr[2, 0]",0.49,0.06,0.37,0.60,2530.24,3854.26,1.00,0.00,0.0
"chol_shot_type_corr[2, 1]",0.53,0.04,0.45,0.62,4434.43,5476.26,1.00,0.00,0.0


In [35]:
corr = trace_chol.posterior["chol_shot_type_corr"]
corr_mean = corr.mean(dim=["chain", "draw"]).values
pd.DataFrame(
    corr_mean,
    index=shot_type_lookup.keys(),
    columns=shot_type_lookup.keys()
)

,DUNK,JUMPER,LAYUP,HOOK
DUNK,1.000000,0.103095,0.489098,0.208735
JUMPER,0.103095,1.000000,0.534644,0.493047
LAYUP,0.489098,0.534644,1.000000,0.379477
HOOK,0.208735,0.493047,0.379477,1.000000


In [38]:
corr_05 = corr.quantile(0.05, dim=("chain", "draw")).values
corr_95 = corr.quantile(0.95, dim=("chain", "draw")).values
corr_05_df = pd.DataFrame(
    corr_05,
    index=shot_type_lookup.keys(),
    columns=shot_type_lookup.keys()
)
corr_95_df = pd.DataFrame(
    corr_95,
    index=shot_type_lookup.keys(),
    columns=shot_type_lookup.keys()
)

In [39]:
corr_05_df

,DUNK,JUMPER,LAYUP,HOOK
DUNK,1.000000,-0.017725,0.387933,0.019321
JUMPER,-0.017725,1.000000,0.461155,0.314595
LAYUP,0.387933,0.461155,1.000000,0.211950
HOOK,0.019321,0.314595,0.211950,1.000000


In [40]:
corr_95_df

,DUNK,JUMPER,LAYUP,HOOK
DUNK,1.000000,0.222052,0.583256,0.388740
JUMPER,0.222052,1.000000,0.604403,0.656254
LAYUP,0.583256,0.604403,1.000000,0.534465
HOOK,0.388740,0.656254,0.534465,1.000000


In [29]:
azs.summary(trace_chol, var_names=["mu"], ci_prob=0.95, round_to=2)

,mean,sd,eti95_lb,eti95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
"mu[A.J. Lawson (1630639), DUNK]",-0.19,0.28,-0.74,0.37,17600.11,5962.36,1.0,0.0,0.0
"mu[A.J. Lawson (1630639), JUMPER]",-0.10,0.09,-0.28,0.08,18101.69,6597.52,1.0,0.0,0.0
"mu[A.J. Lawson (1630639), LAYUP]",-0.13,0.12,-0.36,0.09,16543.77,6944.62,1.0,0.0,0.0
"mu[A.J. Lawson (1630639), HOOK]",-0.09,0.19,-0.47,0.28,14534.80,6516.75,1.0,0.0,0.0
"mu[AJ Green (1631260), DUNK]",0.05,0.36,-0.67,0.75,16935.92,7075.46,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
"mu[Ziaire Williams (1630533), HOOK]",-0.11,0.18,-0.46,0.26,15409.49,4917.14,1.0,0.0,0.0
"mu[Zion Williamson (1629627), DUNK]",0.02,0.17,-0.30,0.36,18876.40,5996.34,1.0,0.0,0.0
"mu[Zion Williamson (1629627), JUMPER]",0.00,0.07,-0.14,0.14,16881.69,6492.43,1.0,0.0,0.0
"mu[Zion Williamson (1629627), LAYUP]",0.17,0.05,0.08,0.26,14684.33,6786.67,1.0,0.0,0.0


In [35]:
coords = {
    "obs_id": np.arange(len(target)),
    "shot_type":shot_type_lookup.keys(),
    "player": player_lookup.keys(),
    "season": season_list,
    }
n_shot_types = len(shot_type_lookup.keys())
n_seasons = len(coords["season"])

with pm.Model(coords=coords) as model:
    shots_made = pm.Data("shots_made", target, dims=("obs_id",))
    offset_data = pm.Data("offset_data", offset_vals, dims=("obs_id",))
    attempts_data = pm.Data("attempts_data", attempts_vals, dims=("obs_id",))
    player_idx = pm.Data("player_idx", player_idx_vals, dims=("obs_id",))
    season_idx = pm.Data("season_idx", season_idx_vals, dims=("obs_id",))
    shot_type_idx = pm.Data("shot_type_idx", shot_type_idx_vals, dims=("obs_id",))

    # baseline
    offset_weight = pm.Normal("offset_weight", mu=0.2, sigma=0.2, dims=("shot_type",))
    shot_type_intercept = pm.Normal("shot_type_intercept", mu=0, sigma=1, dims=("shot_type",))
    # career skill
    player_st_dev = pm.HalfNormal("player_st_dev", 0.25, dims=("shot_type",))
    z_mu = pm.Normal("z_mu", mu=0, sigma=1, dims=("shot_type", "player"))
    mu = pm.Deterministic("mu", z_mu.T * player_st_dev[None, :], dims=("player", "shot_type"))
    # chol, corr, stds = pm.LKJCholeskyCov("chol_shot_type", n=n_shot_types, eta=2.0, sd_dist=pm.Exponential.dist(1.0))
    # z_mu = pm.ZeroSumNormal("z_mu", sigma=1.0, dims=("shot_type", "player"))
    # mu = pm.Deterministic("mu", (chol @ z_mu).T, dims=("player", "shot_type")) # correlated player skill component
    # seasonal component
    sigma_dev = pm.Exponential("sigma_dev", 1.0, dims="shot_type") # season deviation
    delta_raw = pm.ZeroSumNormal(
        "delta_raw",
        sigma=1.0,
        dims=("player", "shot_type", "season"),
        n_zerosum_axes=1
    )
    delta = pm.Deterministic(
        "delta",
        delta_raw * sigma_dev[None, :, None],
        dims=("player", "shot_type", "season")
    )
    skill_grid = mu[:, :, None] + delta
    skill_obs = skill_grid[player_idx, shot_type_idx, season_idx]
    
    theta = (
        offset_weight[shot_type_idx] * offset_data
        + shot_type_intercept[shot_type_idx]
        + skill_obs
    )
    pm.Binomial("obs", n=attempts_data, p=pm.math.sigmoid(theta), observed=shots_made)

In [36]:
trace = sample(model, tune=1000, draws=1000, target_accept=0.9, dense_mass=True)

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


In [37]:
azs.summary(trace, var_names=["offset_weight", "shot_type_intercept", "player_st_dev"], ci_prob=0.95, round_to=2)

,mean,sd,eti95_lb,eti95_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
offset_weight[DUNK],0.56,0.08,0.40,0.73,2331.18,2304.53,1.0,0.0,0.0
offset_weight[JUMPER],0.82,0.07,0.68,0.95,1500.51,1858.73,1.0,0.0,0.0
offset_weight[LAYUP],0.40,0.04,0.32,0.48,2000.20,2495.28,1.0,0.0,0.0
offset_weight[HOOK],0.20,0.13,-0.06,0.47,4470.99,3195.93,1.0,0.0,0.0
shot_type_intercept[DUNK],1.52,0.08,1.37,1.67,2204.07,2480.03,1.0,0.0,0.0
shot_type_intercept[JUMPER],-0.14,0.03,-0.20,-0.07,1401.26,1770.82,1.0,0.0,0.0
shot_type_intercept[LAYUP],0.05,0.02,0.01,0.09,1606.73,2289.51,1.0,0.0,0.0
shot_type_intercept[HOOK],-0.10,0.03,-0.15,-0.05,2423.74,2844.27,1.0,0.0,0.0
player_st_dev[DUNK],0.37,0.02,0.33,0.42,1440.15,1989.75,1.0,0.0,0.0
player_st_dev[JUMPER],0.14,0.01,0.12,0.15,1153.85,1658.81,1.0,0.0,0.0


In [ ]:
df['player_age'].describe()

In [ ]:
age_z = df['age_z'].values

# compute initial params
m_init, c_init = pm.gp.hsgp_approx.approx_hsgp_hyperparams(
    x_range=[np.min(age_z), np.max(age_z)],
    lengthscale_range=[0.75, 1.5],
    cov_func="expquad",
)
print(f"m: {m_init}, c: {c_init:.2f}")

In [ ]:
coords = {
    "obs_id": np.arange(len(target)),
    "shot_type":shot_type_lookup.keys(),
    "player": player_lookup.keys(),
    "season" : ['2021/2022','2022/2023','2023/2024','2024/2025','2025/2026'],
    "season_walk" : ['2022/2023','2023/2024','2024/2025','2025/2026'],
    }
n_shot_types = len(shot_type_lookup.keys())

with pm.Model(coords=coords) as gp_model:
    shots_made = pm.Data("shots_made", target, dims=("obs_id",))
    offset_data = pm.Data("offset_data", offset_vals, dims=("obs_id",))
    attempts_data = pm.Data("attempts_data", attempts_vals, dims=("obs_id",))
    expected_makes_data = pm.Data("expected_makes_data", expected_makes_vals, dims=("obs_id",))
    player_idx = pm.Data("player_idx", player_idx_vals, dims=("obs_id",))
    season_idx = pm.Data("season_idx", season_idx_vals, dims=("obs_id",))
    shot_type_idx = pm.Data("shot_type_idx", shot_type_idx_vals, dims=("obs_id",))
    age_data = pm.Data("age_data", age_z, dims=("obs_id",))

    # baseline
    offset_weight = pm.Normal("offset_weight", mu=1.0, sigma=0.25)
    shot_type_intercept = pm.Normal("shot_type_intercept", mu=0, sigma=1, dims=("shot_type",))
    # career talent
    sigma_career = pm.HalfNormal("sigma_career", sigma=1.0, dims=("shot_type",))
    player_mean_dev = pm.Normal("player_mean_dev", mu=0.0, sigma=1.0, dims=("player", "shot_type"))
    player_mean_centered = player_mean_dev - player_mean_dev.mean(axis=0)
    player_mean = pm.Deterministic(
        "player_mean",
        player_mean_centered * sigma_career[None, :], dims=("player", "shot_type")
        )
    # Mv Random Walk
    L, corr, stds = pm.LKJCholeskyCov("L", n=n_shot_types, eta=2.0, sd_dist=pm.Exponential.dist(beta=1.0))
    z_innovations = pm.Normal(
        "z_innovations",
        mu=0.0, sigma=1.0,
        dims=("season_walk", "player", "shot_type")
    )
    player_innovations = pt.dot(z_innovations, L.T)
    raw_walk = pt.concatenate([
       pt.zeros((1, len(coords["player"]), len(coords["shot_type"]))),
       pt.cumsum(player_innovations, axis=0)
    ], axis=0)
    player_walk_centered = raw_walk - raw_walk.mean(axis=0, keepdims=True)
    player_walk = pm.Deterministic("player_walk", player_walk_centered, dims=("season", "player", "shot_type"))
    # player effect
    player_effect = pm.Deterministic(
        "player_effect",
        player_mean[None, :, :] + player_walk, dims=("season", "player", "shot_type")
        )
    player_contribution = player_effect[season_idx, player_idx, shot_type_idx]

    # Gaussian Process for age
    mean = pm.gp.mean.Zero()
    eta = pm.HalfNormal("eta", sigma=0.5)
    ls = pm.InverseGamma("ls", alpha=3.0, beta=2.0)
    cov_func = eta**2 * pm.gp.cov.ExpQuad(input_dim=1, ls=ls)
    gp = pm.gp.HSGP(m=[m_init], c=c_init, drop_first=True, mean_func=mean, cov_func=cov_func)
    f = gp.prior("f", X=age_data)

    # expected value
    theta = shot_type_intercept[shot_type_idx] + offset_weight*offset_data + player_contribution + f
    pm.Binomial("obs", n=attempts_data, p=pm.math.sigmoid(theta), observed=shots_made)

In [ ]:
samps = 20
with model:
    prior_samples = pm.sample_prior_predictive(samples=samps, var_names=["eta", "ls", "f"], random_seed=RANDOM_SEED)

# gp draws directly from HSGP
gp_draws = prior_samples.prior["f"].values.reshape(-1, len(age_z))
plt.figure(figsize=(8, 5))
for draw in gp_draws:
    sort_idx = np.argsort(age_z)
    plt.plot(age_z[sort_idx], draw[sort_idx], alpha=0.2)
#plt.ylim(-3, 3)
plt.xlabel("age_z")
plt.ylabel("???")
plt.title("Prior Predictive GP Draws")

In [ ]:
gp_trace = sample(gp_model, draws = 2000, tune = 2000, target_accept = 0.95)

In [ ]:
azs.summary(trace, var_names=['eta', 'ls', "offset_weight","shot_type_intercept", "sigma_career", "L"], ci_prob=0.95, round_to=3)

In [ ]:
f_draws = trace.posterior["f"].values
f = f_draws.reshape(-1, f_draws.shape[-1])

curves = f
sort_idx = np.argsort(age_z)
age_sorted = age_z[sort_idx]
curves_sorted = curves[:, sort_idx]

fig, ax = plt.subplots(figsize=(8, 6))
rng = np.random.default_rng(RANDOM_SEED)
subset_indices = rng.choice(curves_sorted.shape[0], size=500, replace=False)
for idx in subset_indices:
    ax.plot(age_sorted, curves_sorted[idx, :], color="darkred", alpha=0.05)

ax.set_title("Aging Curve")
ax.set_xlabel("Age (Standardized)")
ax.set_ylabel("???")
#ax.set_ylim(-3, 3)
ax.grid(True)

In [ ]:
# coords = {
#     "obs_id": np.arange(len(target)),
#     "shot_type":shot_type_lookup.keys(),
#     "player": player_lookup.keys(),
#     "season" : ['2021/2022','2022/2023','2023/2024','2024/2025','2025/2026'],
#     }
# n_shot_types = len(shot_type_lookup.keys())

# with pm.Model(coords=coords) as model:
#     shots_made = pm.Data("shots_made", target, dims=("obs_id",))
#     offset_data = pm.Data("offset_data", offset_vals, dims=("obs_id",))
#     attempts_data = pm.Data("attempts_data", attempts_vals, dims=("obs_id",))
#     player_idx = pm.Data("player_idx", player_idx_vals, dims=("obs_id",))
#     season_idx = pm.Data("season_idx", season_idx_vals, dims=("obs_id",))
#     shot_type_idx = pm.Data("shot_type_idx", shot_type_idx_vals, dims=("obs_id",))

#     # # baseline
#     # offset_weight = pm.Normal("offset_weight", mu=1.0, sigma=0.25)
#     shot_type_intercept = pm.Normal("shot_type_intercept", mu=0, sigma=1, dims=("shot_type",))
#     # Mv Career Skill
#     chol, corr, stds = pm.LKJCholeskyCov("career_cov", n=n_shot_types, eta=2.0, sd_dist=pm.HalfNormal.dist(sigma=0.3))


#     sigma_career = pm.HalfNormal("sigma_career", sigma=0.3, dims=("shot_type",))
#     player_inital = pm.ZeroSumNormal(
#         "player_inital",
#         sigma=0.3,
#         dims=("shot_type", "player"),
#         n_zerosum_axes=1
#     )
#     player_mean = pm.Deterministic(
#         "player_mean",
#         player_inital.T*sigma_career[None, :], dims=("player", "shot_type")
#         )
#     initial_state = player_mean[None, :, :]
    
#     z_player = pm.Normal("z_player", mu=0, sigma=1.0, dims=("player", "shot_type"))
#     career_skill = pm.Deterministic("career_skill", z_player @ chol.T, dims=("player", "shot_type"))
    

#     player_effect = pm.Deterministic(
#         "player_effect",
#         pt.concatenate([initial_state, initial_state + cuml_innovations], axis=0),
#         dims=("season", "player", "shot_type")
#     )
#     player_contribution = player_effect[season_idx, player_idx, shot_type_idx]

#     # expected value
#     theta = shot_type_intercept[shot_type_idx] + player_contribution #  + offset_weight*offset_data
#     pm.Binomial("obs", n=attempts_data, p=pm.math.sigmoid(theta), observed=shots_made)